# 07. New Approaches: Score-Driven Quantization (README 평가식 반영)

## 핵심 원칙 (대회 공식)
- `Score = max(0.5 × PerfNorm + 0.5 × SpeedNorm, 0)`
- `PerfNorm = Perf_model / Perf_base_model`
- `SpeedNorm = 1 - TPOT_model / TPOT_base_model`
- 즉, **정확도/속도 둘 다 50:50**로 동일하게 중요함

## 기존 리포트 기반 교훈
- 서버 최고점은 **FP8 Dynamic base (0.61)**
- 모델 변형이 커질수록 서버 hidden benchmark에서 점수 하락 경향
- 따라서 이번 시도는 **저변형(quantization-only)** + **점수식 직접 최적화**에 집중

## 검색 기반 기술 근거 (공식 문서)
- vLLM은 `compressed-tensors`에서 **FP8 W8A8 / INT8 W8A8 / INT8 W8A16**를 지원
- vLLM FP8 문서: dynamic quantization은 정확도/유연성 장점이 있지만 per-token scale 계산 오버헤드가 있어 latency 이득이 제한될 수 있음
- NVIDIA L4(평가 GPU)는 FP8/INT8 Tensor Core를 지원

## 이번 실험 구성

| Exp | 방법 | 목표 축 | 가설 |
|-----|------|--------|------|
| 0 | FP8 Dynamic | 안정 기준선 | 서버 최고점 재현 |
| 1 | INT8 W8A8 Dynamic | 속도 축 | FP8 대비 속도 우위 가능성 확인 |
| 2 | FP8 Static (MANTA 256) | 정확도+속도 균형 | 동적 스케일 오버헤드 제거 + 보수적 캘리브레이션 |
| 3 | FP8 Static (Mixed 512) | 일반화 | 캘리브레이션 분포 확장으로 hidden set 대응 |
| 4 | FP8 Static (MANTA 512) | 정확도 축 | in-domain 샘플 증가로 정적 스케일 안정화 |

## 서버 제약 (README)
- vLLM 0.14.1 고정 (수정 불가)
- `lm_head` 양자화 불가 → `ignore=["lm_head"]` 유지
- 제출물은 HF `model/` 폴더만 가능
- 전체 실행 20분 제한


---
## 0. 공통 설정 / 유틸리티


In [ ]:
import os
import json
import shutil
import time
import glob
import random
import numpy as np
import torch
import gc
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset, concatenate_datasets
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier

# ── 경로 ──
BASE_MODEL = "${PROJECT_ROOT}/open/base_model"
BASE_DIR = "${PROJECT_ROOT}/07_new_approaches/code"
EVAL_SCRIPT = "${PROJECT_ROOT}/eval/run_eval.py"
EVAL_JSON_GLOB = "${PROJECT_ROOT}/eval/eval_*.json"

# ── 상수 ──
SEED = 42
TORCH_DTYPE = torch.float16
BASE_IGNORE = ["lm_head"]  # vLLM Exaone4: lm_head는 Embedding 타입이라 양자화 불가
MAX_SEQ_LENGTH = 2048

os.makedirs(BASE_DIR, exist_ok=True)
print(f"BASE_MODEL: {BASE_MODEL}")
print(f"BASE_DIR:   {BASE_DIR}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")
else:
    print("GPU not available")


In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_model_and_tokenizer():
    """기본 모델 로드 (FP16)."""
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=TORCH_DTYPE,
        device_map="auto",
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    return model, tokenizer


def save_model(model, tokenizer, output_dir):
    """모델 저장 + chat_template 복사 + 필수 파일 검증."""
    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    src_template = os.path.join(BASE_MODEL, "chat_template.jinja")
    if os.path.exists(src_template):
        shutil.copy2(src_template, os.path.join(output_dir, "chat_template.jinja"))

    required = ["config.json", "tokenizer.json", "tokenizer_config.json"]
    for f in required:
        assert os.path.exists(os.path.join(output_dir, f)), f"{f} missing!"

    print(f"  저장 완료: {output_dir}")


def cleanup_model(model):
    """GPU 메모리 해제."""
    if model is not None:
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def create_submission_zip(model_dir, zip_name):
    """submit.zip 생성 (model/ 폴더 구조)."""
    parent = os.path.dirname(model_dir)
    basename = os.path.basename(model_dir)
    zip_path = shutil.make_archive(
        base_name=os.path.join(parent, zip_name),
        format="zip",
        root_dir=parent,
        base_dir=basename,
    )
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"  ZIP: {zip_path} ({size_mb:.1f} MB)")
    return zip_path


def prepare_manta_calib(tokenizer, num_samples=256):
    """MANTA-1M 캘리브레이션 데이터 (대화형)."""
    ds = load_dataset("LGAI-EXAONE/MANTA-1M", split="train")
    ds = ds.shuffle(seed=SEED).select(range(num_samples))

    def preprocess(example):
        return {
            "text": tokenizer.apply_chat_template(
                example["conversations"],
                add_generation_prompt=True,
                tokenize=False,
            )
        }

    ds = ds.map(preprocess, num_proc=4, remove_columns=ds.column_names)
    print(f"  MANTA calibration: {len(ds)} samples")
    return ds


def prepare_mixed_calib(tokenizer, num_per_source=256):
    """MANTA + pile-10k 혼합 캘리브레이션 데이터."""
    manta = load_dataset("LGAI-EXAONE/MANTA-1M", split="train")
    manta = manta.shuffle(seed=SEED).select(range(num_per_source))

    def preprocess_manta(example):
        return {
            "text": tokenizer.apply_chat_template(
                example["conversations"],
                add_generation_prompt=True,
                tokenize=False,
            )
        }

    manta = manta.map(preprocess_manta, num_proc=4, remove_columns=manta.column_names)

    try:
        pile = load_dataset("NeelNanda/pile-10k", split="train")
        pile = pile.shuffle(seed=SEED).select(range(num_per_source))
        pile = pile.select_columns(["text"])
        combined = concatenate_datasets([manta, pile]).shuffle(seed=SEED)
        print(
            f"  Mixed calibration: {len(combined)} samples "
            f"(MANTA {num_per_source} + pile {num_per_source})"
        )
    except Exception as e:
        print(f"  pile-10k 로드 실패 ({e}), MANTA만 사용")
        combined = manta

    return combined


def run_quant_experiment(
    exp_name,
    scheme,
    calib_ds=None,
    num_calibration_samples=None,
    max_seq_length=MAX_SEQ_LENGTH,
):
    """단일 양자화 실험 실행 후 model 저장 + submit.zip 생성."""
    out_dir = os.path.join(BASE_DIR, exp_name, "model")

    print("=" * 60)
    print(f"{exp_name}: scheme={scheme} -> {out_dir}")
    if calib_ds is None:
        print("  mode: data-free")
    else:
        print(f"  mode: calibration ({num_calibration_samples} samples)")
    print("=" * 60)

    set_seed()
    model = None
    try:
        model, tokenizer = load_model_and_tokenizer()
        recipe = QuantizationModifier(
            targets="Linear",
            scheme=scheme,
            ignore=BASE_IGNORE,
        )

        start = time.time()
        if calib_ds is None:
            oneshot(model=model, recipe=recipe)
        else:
            oneshot(
                model=model,
                dataset=calib_ds,
                recipe=recipe,
                max_seq_length=max_seq_length,
                num_calibration_samples=num_calibration_samples,
            )
        print(f"  양자화 완료: {time.time() - start:.1f}s")

        save_model(model, tokenizer, out_dir)
        zip_path = create_submission_zip(out_dir, f"submit_{exp_name}")
        return out_dir, zip_path
    finally:
        cleanup_model(model)


def compute_competition_score(perf_model, perf_base, tpot_model, tpot_base):
    """README 평가식 그대로 계산."""
    perf_norm = perf_model / perf_base
    speed_norm = 1.0 - (tpot_model / tpot_base)
    score = max(0.5 * perf_norm + 0.5 * speed_norm, 0.0)
    return {
        "perf_norm": perf_norm,
        "speed_norm": speed_norm,
        "score": score,
    }


def required_speed_norm(perf_norm, target_score):
    """주어진 PerfNorm에서 목표 Score를 넘기기 위한 최소 SpeedNorm."""
    return (2.0 * target_score) - perf_norm


print("유틸리티 함수 로드 완료")


---
## Exp0. FP8 Dynamic Base (서버 최고점 재현)

- `scheme="FP8_DYNAMIC"`
- data-free, 캘리브레이션 불필요
- 기존 서버 최고점(0.61) 기준선 재현


In [ ]:
EXP0_NAME = "exp0_fp8_dynamic_base"
EXP0_DIR, EXP0_ZIP = run_quant_experiment(
    exp_name=EXP0_NAME,
    scheme="FP8_DYNAMIC",
)


---
## Exp1. INT8 W8A8 Dynamic (속도 축 도전)

- `scheme="INT8"`
- data-free
- FP8 대비 정확도 손실을 최소화하면서 속도 우위를 노리는 후보


In [ ]:
EXP1_NAME = "exp1_int8_w8a8_dynamic"
EXP1_DIR, EXP1_ZIP = run_quant_experiment(
    exp_name=EXP1_NAME,
    scheme="INT8",
)


---
## Exp2. FP8 Static (MANTA 256)

- `scheme="FP8"` (static)
- calibration: MANTA 256
- dynamic scale 계산 오버헤드 제거 + 보수적(in-domain) 캘리브레이션


In [ ]:
EXP2_NAME = "exp2_fp8_static_manta256"

set_seed()
tok_for_calib = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
calib_manta_256 = prepare_manta_calib(tok_for_calib, num_samples=256)

EXP2_DIR, EXP2_ZIP = run_quant_experiment(
    exp_name=EXP2_NAME,
    scheme="FP8",
    calib_ds=calib_manta_256,
    num_calibration_samples=256,
)


---
## Exp3/4. FP8 Static 추가 변형

### Exp3: Mixed 512
- MANTA 256 + pile-10k 256
- hidden benchmark 분포 대응을 위한 calibration 다양화

### Exp4: MANTA 512
- in-domain 샘플 수를 2배로 늘려 static scale 안정화


In [ ]:
# 점수식 감도 점검: PerfNorm이 떨어질 때 필요한 SpeedNorm
TARGET_SCORE = 0.61  # 현재 서버 최고점(참고)
perf_grid = [0.80, 0.85, 0.90, 0.95, 1.00]

print(f"목표 Score: {TARGET_SCORE}")
print("PerfNorm -> 최소 필요 SpeedNorm")
for p in perf_grid:
    need = required_speed_norm(p, TARGET_SCORE)
    print(f"  {p:.2f} -> {need:+.3f}")

print()
print("해석:")
print("- PerfNorm이 낮아질수록 매우 큰 속도 이득이 필요")
print("- hidden benchmark 불확실성이 크므로 저변형 + 정확도 보존이 유리")


### Exp3. FP8 Static (Mixed 512)


In [ ]:
EXP3A_NAME = "exp3_fp8_static_mixed512"

set_seed()
tok_for_mixed = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
calib_mixed_512 = prepare_mixed_calib(tok_for_mixed, num_per_source=256)

EXP3A_DIR, EXP3A_ZIP = run_quant_experiment(
    exp_name=EXP3A_NAME,
    scheme="FP8",
    calib_ds=calib_mixed_512,
    num_calibration_samples=512,
)


### Exp4. FP8 Static (MANTA 512)


In [ ]:
EXP3B_NAME = "exp4_fp8_static_manta512"

set_seed()
tok_for_manta_512 = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
calib_manta_512 = prepare_manta_calib(tok_for_manta_512, num_samples=512)

EXP3B_DIR, EXP3B_ZIP = run_quant_experiment(
    exp_name=EXP3B_NAME,
    scheme="FP8",
    calib_ds=calib_manta_512,
    num_calibration_samples=512,
)


---
## 4. 로컬 평가 (대회식 프록시 계산용)

중요 변경점:
- **기준 모델(index=0)을 원본 `BASE_MODEL`로 고정**
- 이후 PerfNorm/SpeedNorm을 README 공식과 동일한 형태로 계산
- 로컬의 시간은 TPOT 완전 동일값은 아니므로 **TPOT proxy**로 해석


In [ ]:
# 평가 대상 모델 수집 (index=0은 반드시 원본 base model)
eval_models = [BASE_MODEL]
model_names = ["base_fp16"]

candidates = [
    (EXP0_NAME, EXP0_DIR),
    (EXP1_NAME, EXP1_DIR),
    (EXP2_NAME, EXP2_DIR),
    (EXP3A_NAME, EXP3A_DIR),
    (EXP3B_NAME, EXP3B_DIR),
]

for name, path in candidates:
    if os.path.exists(os.path.join(path, "config.json")):
        eval_models.append(path)
        model_names.append(name)
        print(f"  [✓] {name}")
    else:
        print(f"  [✗] {name} (없음 — 해당 셀을 먼저 실행)")

eval_paths = " ".join(eval_models)
eval_cmd = (
    f"python {EVAL_SCRIPT} "
    f"--model_path {eval_paths} "
    f"--n_runs 3 "
    f"--baseline_model_idx 0 "
    f"--tasks gsm8k,mmlu,arc_challenge "
    f"--limit 512 "
    f"--output_dir ${PROJECT_ROOT}/07_new_approaches/eval_results"
)

print()
print(f"평가 대상: {len(eval_models)}개 모델")
print("  index=0 고정: base_fp16")
print()
print("평가 명령어:")
print(eval_cmd)


In [ ]:
# 평가 실행 (시간이 오래 걸리므로 필요시 주석 해제)
# !{eval_cmd}


---
## 5. 제출 ZIP + 점수식 프록시 리포트


In [ ]:
# model/ 폴더를 submit_exp3_fp8_static_mixed512.zip으로 압축
from pathlib import Path
import zipfile

MODEL_DIR = Path("${PROJECT_ROOT}/07_new_approaches/code/exp3_fp8_static_mixed512/model")
ZIP_PATH = MODEL_DIR.parent / "submit_exp3_fp8_static_mixed512.zip"

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"model 폴더가 없습니다: {MODEL_DIR}")

# 기존 zip 있으면 삭제 후 재생성
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in MODEL_DIR.rglob("*"):
        if p.is_file():
            # ZIP 내부 경로가 model/... 형태가 되도록 유지
            arcname = p.relative_to(MODEL_DIR.parent).as_posix()
            zf.write(p, arcname)

# 검증
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = zf.namelist()
    top_entries = sorted({n.split("/")[0] for n in names if n})
    print("ZIP 내부 최상위:", top_entries)
    print("파일 수:", len(names))
    print("앞 10개:", names[:10])

size_mb = ZIP_PATH.stat().st_size / (1024 * 1024)
print(f"\n생성 완료: {ZIP_PATH}")
print(f"크기: {size_mb:.1f} MB")


---
## 6. 제출 전략 (1일 3회 기준)

1. `score_proxy` 상위 2개 + 안정형(FP8 Dynamic base) 1개 제출
2. static 계열이 `PerfNorm`과 `SpeedNorm`을 동시에 개선하면 우선 제출
3. hidden benchmark 분포 리스크를 고려해 **고변형 학습(KD/과도한 FT) 모델은 제외**
